In [8]:
pip install ta_lib

In [9]:
import pandas as pd
import numpy as np
import yfinance as yf
import talib
df = yf.download('BTC-USD', start='2020-01-01', end='2025-01-01')
df = df.droplevel('Ticker', axis=1)
df


/tmp/ipython-input-2037991366.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('BTC-USD', start='2020-01-01', end='2025-01-01')
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Date,,,,,
2020-01-01,7200.174316,7254.330566,7174.944336,7194.892090,18565664997
2020-01-02,6985.470215,7212.155273,6935.270020,7202.551270,20802083465
2020-01-03,7344.884277,7413.715332,6914.996094,6984.428711,28111481032
2020-01-04,7410.656738,7427.385742,7309.514160,7345.375488,18444271275
2020-01-05,7411.317383,7544.497070,7400.535645,7410.451660,19725074095
...,...,...,...,...,...
2024-12-27,94164.859375,97294.843750,93310.742188,95704.976562,52419934565
2024-12-28,95163.929688,95525.898438,94014.289062,94160.187500,24107436185
2024-12-29,93530.226562,95174.875000,92881.789062,95174.054688,29635885267


In [10]:
df['ret'] = df['Close'].pct_change()

df = df.sort_index()



# 1. Simple Moving Average (SMA) of returns (5-period)
df['SMA_ret_5'] = talib.SMA(df['ret'], timeperiod=5)

# 2. Exponential Moving Average (EMA) of returns (5-period)
df['EMA_ret_5'] = talib.EMA(df['ret'], timeperiod=5)

# 3. Standard Deviation (STD) of returns (5-period)
df['STD_ret_5'] = talib.STDDEV(df['ret'], timeperiod=5, nbdev=1)

# 4. Relative Strength Index (RSI) adapted for returns (14-period)
# RSI typically uses price data, but we can compute it on returns
df['RSI_ret_14'] = talib.RSI(df['ret'], timeperiod=14)

# 5. Average True Range (ATR) (14-period, using High, Low, Close)
df['ATR_14'] = talib.ATR(df['High'], df['Low'], df['Close'], timeperiod=14)

# 6. Average Directional Index (ADX) (14-period, using High, Low, Close)
df['ADX_14'] = talib.ADX(df['High'], df['Low'], df['Close'], timeperiod=14)

# 7. Commodity Channel Index (CCI) (14-period, using High, Low, Close)
df['CCI_14'] = talib.CCI(df['High'], df['Low'], df['Close'], timeperiod=14)

# 8. Rate of Change (ROC) of returns (5-period)
df['ROC_ret_5'] = talib.ROC(df['ret'], timeperiod=5)

# 9. Bollinger Bands Width (based on returns, 20-period SMA, 2 STD)
upper_band, middle_band, lower_band = talib.BBANDS(df['ret'], timeperiod=20, nbdevup=2, nbdevdn=2, matype=0)
df['BB_Width_20'] = (upper_band - lower_band) / middle_band

# 10. Momentum of returns (5-period)
df['Momentum_ret_5'] = talib.MOM(df['ret'], timeperiod=5)

df

Price,Close,High,Low,Open,Volume,ret,SMA_ret_5,EMA_ret_5,STD_ret_5,RSI_ret_14,ATR_14,ADX_14,CCI_14,ROC_ret_5,BB_Width_20,Momentum_ret_5
Date,,,,,,,,,,,,,,,,
2020-01-01,7200.174316,7254.330566,7174.944336,7194.892090,18565664997,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-02,6985.470215,7212.155273,6935.270020,7202.551270,20802083465,-0.029819,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-03,7344.884277,7413.715332,6914.996094,6984.428711,28111481032,0.051452,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-04,7410.656738,7427.385742,7309.514160,7345.375488,18444271275,0.008955,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-05,7411.317383,7544.497070,7400.535645,7410.451660,19725074095,0.000089,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-27,94164.859375,97294.843750,93310.742188,95704.976562,52419934565,-0.017022,-0.001651,-0.010434,0.025907,48.096742,4167.081190,26.569075,-87.437834,-21.926918,-38.556537,0.004781
2024-12-28,95163.929688,95525.898438,94014.289062,94160.187500,24107436185,0.010610,0.001351,-0.003419,0.026282,51.482058,3977.404632,26.096471,-79.629707,-340.997289,-36.689747,0.015012
2024-12-29,93530.226562,95174.875000,92881.789062,95174.054688,29635885267,-0.017167,-0.010510,-0.008002,0.016909,48.086689,3857.096154,25.917582,-92.736049,-140.740928,-55.774124,-0.059305


In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Assume df is your DataFrame (load via pd.read_csv if needed)
# Create target: daily range
df['range'] = df['High'].shift(-1) - df['Low'].shift(-1)

# Features from your data (after 'regime')
features = ['ret', 'SMA_ret_5', 'EMA_ret_5', 'STD_ret_5', 'RSI_ret_14', 'ATR_14',
            'ADX_14', 'CCI_14', 'ROC_ret_5', 'BB_Width_20', 'Momentum_ret_5']

# Drop NaNs (early rows may have them)
df = df[features + ['range']].dropna()

# Split data: Use time-based split (80% train, 20% test)
split_idx = int(len(df) * 0.8)
X_train = df[features].iloc[:split_idx]
X_test = df[features].iloc[split_idx:]
y_train = df['range'].iloc[:split_idx]
y_test = df['range'].iloc[split_idx:]

# Scale features (optional but recommended for consistency)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Predict and evaluate
y_pred = model.predict(X_test_scaled)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error: {mse:.4f}")
print(f"R² Score: {r2:.4f}")

# Feature importances
importances = pd.DataFrame({'Feature': features, 'Importance': model.feature_importances_}).sort_values('Importance', ascending=False)
print(importances)

# Example: Predict for new data (e.g., a future row on 2025-09-30)
new_data = pd.DataFrame({  # Replace with actual values
    'ret': [0.005], 'SMA_ret_5': [0.002], 'EMA_ret_5': [0.0015], 'STD_ret_5': [0.008],
    'RSI_ret_14': [55.0], 'ATR_14': [25.0], 'ADX_14': [20.0], 'CCI_14': [-50.0],
    'ROC_ret_5': [-100.0], 'BB_Width_20': [0.01], 'Momentum_ret_5': [0.007]
})
new_scaled = scaler.transform(new_data)
predicted_range = model.predict(new_scaled)[0]
print(f"Predicted (high - low) range: {predicted_range:.4f}")

Mean Squared Error: 2418063.4729
R² Score: 0.1525
           Feature  Importance
5           ATR_14    0.589817
6           ADX_14    0.057001
1        SMA_ret_5    0.052665
3        STD_ret_5    0.045624
4       RSI_ret_14    0.040774
7           CCI_14    0.039911
2        EMA_ret_5    0.039213
10  Momentum_ret_5    0.039107
8        ROC_ret_5    0.036263
9      BB_Width_20    0.033486
0              ret    0.026138
Predicted (high - low) range: 215.8102
